In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
import warnings
import zipfile
from pathlib import Path

DATA = Path('../data/raw')  # ИСПРАВЛЕНО: CSV теперь лежат в data/raw, а не рядом с ноутбуком

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score

In [ ]:
# ИСПРАВЛЕНО: глобальное отключение предупреждений убрано.
# warnings.filterwarnings("ignore") прячет ВСЕ предупреждения pandas/sklearn, в том числе полезные:
# здесь оно скрывало FutureWarning про replace(..., inplace=True) и SettingWithCopyWarning про запись в срез.
# Если какое-то предупреждение прочитано и понятно, отключайте только его категорию:
# warnings.filterwarnings("ignore", category=FutureWarning)

Данные: распакуйте архив Kaggle в `data/raw/` один раз (ячейка с zipfile больше не нужна).

In [ ]:
POS_CASH_balance = pd.read_csv(DATA / 'POS_CASH_balance.csv')
bureau = pd.read_csv(DATA / 'bureau.csv')
bureau_balance = pd.read_csv(DATA / 'bureau_balance.csv')
previous_application = pd.read_csv(DATA / 'previous_application.csv')
installments_payments = pd.read_csv(DATA / 'installments_payments.csv')
credit_card_balance = pd.read_csv(DATA / 'credit_card_balance.csv')

In [ ]:
application_test = pd.read_csv(DATA / 'application_test.csv')
application_train = pd.read_csv(DATA / 'application_train.csv')

In [ ]:
print(POS_CASH_balance.shape)
print(application_test.shape)
print(application_train.shape)
print(bureau.shape)
print(bureau_balance.shape)
print(previous_application.shape)
print(installments_payments.shape)
print(credit_card_balance.shape)

In [ ]:
application_train.info()

In [ ]:
application_test.info()

In [ ]:
print(os.listdir(DATA))

In [ ]:
print(f"Train dataset shape: {application_train.shape}")
application_train.head()

In [ ]:
print(f"Test dataset shape: {application_test.shape}")
application_test.head()

##Exploratory data analysis

In [ ]:
application_train['TARGET'].value_counts()

In [ ]:
application_train.TARGET.astype(int).plot.hist()

Missing values

In [ ]:
def missing_values_table(df):
    missing_values = df.isna().sum()
    # ИСПРАВЛЕНО: было 100 * missing / len(missing_values), то есть деление на ЧИСЛО СТОЛБЦОВ (122),
    # отсюда проценты вида 176118.9. Доля пропусков = среднее от isna() по строкам.
    percentage_of_missing_values = 100 * df.isna().mean()

    table = pd.concat([missing_values, percentage_of_missing_values], axis=1)
    table = table.rename(columns={0: 'Missing Values', 1: '% of Total Values'})
    table = table[table['Missing Values'] != 0].sort_values('% of Total Values', ascending=False).round(1)

    print(f"Your selected dataframe has {df.shape[1]} columns.\nThere are {table.shape[0]} columns that have missing values.")
    return table

In [ ]:
train_missing_values = missing_values_table(application_train)
print(train_missing_values)

Было: проценты пропусков больше 100 000%. Причина в делении на число столбцов вместо числа строк, исправлено выше.

## Column Types

In [ ]:
# Number of each type of column
application_train.dtypes.value_counts()

In [ ]:
# Number of unique classes in each object column
application_train.select_dtypes('object').apply(pd.Series.nunique, axis=0)

## Encoding categorical variables

## Label Encoding and One-Hot Encoding

In [ ]:
def label_encoding(df_train, df_test):
    le = LabelEncoder()
    le_count = 0

    for col in df_train:
        if df_train[col].dtype == 'object':
            if len(list(df_train[col].unique())) <= 2:
                le.fit(df_train[col])      # fit: запомнить список категорий train, например ['N', 'Y'] -> 0, 1
                df_train[col] = le.transform(df_train[col])  # transform: заменить строки кодами по запомненному списку
                df_test[col] = le.transform(df_test[col])    # тот же список для test, иначе коды разъедутся

                le_count += 1

    print(f"{le_count} columns were label encoded")
    return df_train, df_test

In [ ]:
application_train, application_test = label_encoding(application_train, application_test)

## One-hot encoding

In [ ]:
application_train = pd.get_dummies(application_train)
application_test = pd.get_dummies(application_test)

print('Training Features shape: ', application_train.shape)
print('Testing Features shape: ', application_test.shape)

In [ ]:
train_labels = application_train.TARGET

application_train, application_test = application_train.align(application_test, join='inner', axis=1)
application_train['TARGET'] = train_labels

print('Training Features shape: ', application_train.shape)
print('Testing Features shape: ', application_test.shape)

## Back to EDA

## Anomalies

In [ ]:
application_train.columns

In [ ]:
(application_train['DAYS_BIRTH'] / -365).describe()

In [ ]:
application_train['DAYS_EMPLOYED'].describe()

In [ ]:
application_train['DAYS_EMPLOYED'].plot.hist(title = 'Days Employment Histogram')
plt.xlabel('Days Employment')

In [ ]:
anomalies = application_train[application_train['DAYS_EMPLOYED'] == 365243]
non_anomalies = application_train[application_train['DAYS_EMPLOYED'] != 365243]

print(f"The non-anomalies default on {100 * (non_anomalies['TARGET'].mean()):.2f}% of loans")
print(f"The anomalies default on {100 * (anomalies['TARGET'].mean()):.2f}% of loans")
print(f"There are {len(anomalies)} anomalous days of employment")  # ИСПРАВЛЕНО: подпись

In [ ]:
application_train['DAYS_EMPLOYED_anomalies'] = application_train['DAYS_EMPLOYED'] == 365243
application_test['DAYS_EMPLOYED_anomalies'] = application_test['DAYS_EMPLOYED'] == 365243

# ИСПРАВЛЕНО: replace(..., inplace=True) на отдельном столбце в pandas 3 перестанет менять датафрейм.
# Надёжный вариант: присвоить результат обратно.
application_train['DAYS_EMPLOYED'] = application_train['DAYS_EMPLOYED'].replace({365243: np.nan})
application_test['DAYS_EMPLOYED'] = application_test['DAYS_EMPLOYED'].replace({365243: np.nan})

In [ ]:
application_train.DAYS_EMPLOYED.plot.hist(title='Days Employment Histogram')
plt.xlabel('Days Employment')

In [ ]:
print('There are %d anomalies in the test data out of %d entries' % (application_test["DAYS_EMPLOYED_anomalies"].sum(), len(application_test)))

## Correlations

In [ ]:
app_train_corr = application_train.corr()['TARGET'].sort_values()

In [ ]:
print(f"Most Positive Correlations:\n {app_train_corr.tail(15)}")
print(f"Most Negative Correlations:\n {app_train_corr.head(15)}")

In [ ]:
# ИСПРАВЛЕНО: возраст переводится в годы и в train, и в test.
# Раньше test оставался в отрицательных днях, и модель обучалась на 20..70, а предсказывала на -25000..-7000.
application_train['DAYS_BIRTH'] = application_train['DAYS_BIRTH'] / -365
application_test['DAYS_BIRTH'] = application_test['DAYS_BIRTH'] / -365

In [ ]:
application_train['DAYS_BIRTH'].corr(application_train['TARGET'])

In [ ]:
plt.style.use('fivethirtyeight')
sns.histplot(application_train['DAYS_BIRTH'], bins=25, edgecolor='k')

plt.title('Age of Client'); plt.xlabel('Age (years)'); plt.ylabel('Count');

In [ ]:
plt.figure(figsize=(10, 8))

sns.kdeplot(application_train.loc[application_train['TARGET'] == 0, 'DAYS_BIRTH'], label='Target = 0')
sns.kdeplot(application_train.loc[application_train['TARGET'] == 1, 'DAYS_BIRTH'], label='Target = 1')

plt.title('Distribution of Ages'); plt.xlabel('Age (years)'); plt.ylabel('Density'); 

## Years binned

In [ ]:
age_data = application_train[['TARGET', 'DAYS_BIRTH']].copy()  # ИСПРАВЛЕНО: .copy(), иначе запись в срез

age_data['YEARS_BINNED'] = pd.cut(age_data['DAYS_BIRTH'], bins=np.linspace(20, 70, num=11))
age_data.head(10)

In [ ]:
age_groups = age_data.groupby('YEARS_BINNED', observed=False)[['TARGET', 'DAYS_BIRTH']].mean()

age_groups.head(10)

In [ ]:
plt.figure(figsize=(8, 8))

plt.bar(age_groups.index.astype(str), 100 * age_groups.TARGET)
plt.xticks(rotation=75)
plt.title('Failure to Repay by Age Group')
plt.xlabel('Age Group (years)')
plt.ylabel('Failure to Repay (%)')

In [ ]:
ext_data = application_train[['TARGET', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH']]
ext_data_corr = ext_data.corr()
ext_data_corr

In [ ]:
plt.figure(figsize = (10, 12))

for i, col in enumerate(['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']):
    plt.subplot(3, 1, i + 1)
    sns.kdeplot(ext_data.loc[ext_data.TARGET == 1, col])
    sns.kdeplot(ext_data.loc[ext_data.TARGET == 0, col])

## Feature Engineering

## Polynomial features

In [ ]:
poly_features = application_train[['TARGET', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH']]
poly_features_test = application_test[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH']]

poly_target = poly_features['TARGET']
poly_features = poly_features.drop(columns=['TARGET'])

from sklearn.impute import SimpleImputer

# Imputer заполняет пропуски (NaN) статистикой столбца: здесь средним.
imp = SimpleImputer(strategy='mean')

# fit: посчитать и запомнить среднее каждого столбца по TRAIN.
# transform: заполнить NaN этими запомненными средними.
# fit_transform: оба шага подряд на одних данных. Для test только transform: средние берутся из train.
poly_features = imp.fit_transform(poly_features)
poly_features_test = imp.transform(poly_features_test)

from sklearn.preprocessing import PolynomialFeatures

poly_transformer = PolynomialFeatures(degree = 3)

In [ ]:
# Train the polynomial features
poly_transformer.fit(poly_features)

# Transform the features
poly_features = poly_transformer.transform(poly_features)
poly_features_test = poly_transformer.transform(poly_features_test)
print('Polynomial Features shape: ', poly_features.shape)

In [ ]:
poly_transformer.get_feature_names_out(input_features = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH'])[:15]

In [ ]:
poly_features = pd.DataFrame(poly_features, columns=poly_transformer.get_feature_names_out(input_features=['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH']))
poly_features['TARGET'] = poly_target

poly_corrs = poly_features.corr()['TARGET'].sort_values()

# Display most negative and most positive
print(poly_corrs.head(10))
print(poly_corrs.tail(5))

In [ ]:
poly_names = poly_transformer.get_feature_names_out(['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH'])
poly_features_test = pd.DataFrame(poly_features_test, columns=poly_names)

# ИСПРАВЛЕНО: в полиномах есть константа '1' и сами исходные столбцы.
# Без удаления merge создаст дубли EXT_SOURCE_1_x / EXT_SOURCE_1_y.
dup_cols = ['1', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH']
poly_train_new = poly_features.drop(columns=dup_cols + ['TARGET'])
poly_test_new = poly_features_test.drop(columns=dup_cols)

poly_train_new['SK_ID_CURR'] = application_train['SK_ID_CURR'].values
app_train_poly = application_train.merge(poly_train_new, on='SK_ID_CURR', how='left')

poly_test_new['SK_ID_CURR'] = application_test['SK_ID_CURR'].values
app_test_poly = application_test.merge(poly_test_new, on='SK_ID_CURR', how='left')

# align с join='inner' выкидывает TARGET (его нет в test), возвращаем
app_train_poly, app_test_poly = app_train_poly.align(app_test_poly, join='inner', axis=1)
app_train_poly['TARGET'] = application_train['TARGET'].values

print('Training data with polynomial features shape: ', app_train_poly.shape)
print('Testing data with polynomial features shape:  ', app_test_poly.shape)

## Domain knowledge Features

In [ ]:
app_train_domain = application_train.copy()
app_test_domain = application_test.copy()

app_train_domain['CREDIT_INCOME_PERCENT'] = app_train_domain['AMT_CREDIT'] / app_train_domain['AMT_INCOME_TOTAL']
app_train_domain['ANNUITY_INCOME_PERCENT'] = app_train_domain['AMT_ANNUITY'] / app_train_domain['AMT_INCOME_TOTAL']
app_train_domain['CREDIT_TERM'] = app_train_domain['AMT_ANNUITY'] / app_train_domain['AMT_CREDIT']
# ИСПРАВЛЕНО: DAYS_BIRTH уже в годах, поэтому стаж тоже переводим в годы
app_train_domain['DAYS_EMPLOYED_PERCENT'] = (-app_train_domain['DAYS_EMPLOYED'] / 365) / app_train_domain['DAYS_BIRTH']

In [ ]:
app_test_domain['CREDIT_INCOME_PERCENT'] = app_test_domain['AMT_CREDIT'] / app_test_domain['AMT_INCOME_TOTAL']
app_test_domain['ANNUITY_INCOME_PERCENT'] = app_test_domain['AMT_ANNUITY'] / app_test_domain['AMT_INCOME_TOTAL']
app_test_domain['CREDIT_TERM'] = app_test_domain['AMT_ANNUITY'] / app_test_domain['AMT_CREDIT']
app_test_domain['DAYS_EMPLOYED_PERCENT'] = (-app_test_domain['DAYS_EMPLOYED'] / 365) / app_test_domain['DAYS_BIRTH']

In [ ]:
plt.figure(figsize = (12, 20))
# iterate through the new features
for i, feature in enumerate(['CREDIT_INCOME_PERCENT', 'ANNUITY_INCOME_PERCENT', 'CREDIT_TERM', 'DAYS_EMPLOYED_PERCENT']):
    
    # create a new subplot for each source
    plt.subplot(4, 1, i + 1)
    # plot repaid loans
    sns.kdeplot(app_train_domain.loc[app_train_domain['TARGET'] == 0, feature], label = 'target == 0')
    # plot loans that were not repaid
    sns.kdeplot(app_train_domain.loc[app_train_domain['TARGET'] == 1, feature], label = 'target == 1')
    
    # Label the plots
    plt.title('Distribution of %s by Target Value' % feature)
    plt.xlabel('%s' % feature); plt.ylabel('Density');
    
plt.tight_layout(h_pad = 2.5)

## Baselines

In [ ]:
# ИСПРАВЛЕНО: baseline с валидацией. Было: обучение на всём train без метрики,
# SK_ID_CURR как признак, а test масштабировался в обход импутации (отсюда ошибка "Input X contains NaN").
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

X = application_train.drop(columns=['TARGET', 'SK_ID_CURR'])
y = application_train['TARGET']
X_test = application_test[X.columns]

# Pipeline: на каждом фолде imputer и scaler делают fit только на обучающей части
pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', MinMaxScaler()),
    ('model', LogisticRegression(C=0.0001, max_iter=1000)),
])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
oof = cross_val_predict(pipe, X, y, cv=skf, method='predict_proba')[:, 1]
auc = roc_auc_score(y, oof)
print(f"OOF ROC AUC: {auc:.4f} | Gini: {2 * auc - 1:.4f}")
# Попробуйте C = 0.001, 0.01, 0.1: C=0.0001 это очень сильная регуляризация.

In [ ]:
pipe.fit(X, y)  # финальная модель на всём train, только для submission
log_reg_pred = pipe.predict_proba(X_test)[:, 1]

In [ ]:
submit = application_test[['SK_ID_CURR']].copy()  # ИСПРАВЛЕНО: .copy()
submit['TARGET'] = log_reg_pred
submit.head()

Дальше эксперименты идут через `python -m src.train` (см. LEARNING.md), а этот ноутбук остаётся для EDA.